What happens if we bootstrap the fits?

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("..")

In [ ]:
import desi_utils

In [ ]:
import fit_spectrum

In [ ]:
from copy import deepcopy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Data loading

In [ ]:
targetid = 39627677191115285

In [ ]:
props = fit_spectrum.get_properties(targetid)

In [ ]:
spec = fit_spectrum.get_spectrum(props)

In [ ]:
spec_resamp, flux_scale = desi_utils.spectrum.log_rebin_spectrum(spec)

In [ ]:
spec_resamp.plot()

# Bootstrapping

## Fitting a best model

In [ ]:
sedmodel = desi_utils.fit_continuum.SEDModel(
    sps_file = "spectra_fsps_9.0.npz"
)

In [ ]:
ppxf_best = sedmodel.fit(spec_resamp)

In [ ]:
resid = spec_resamp.flux - ppxf_best.bestfit

In [ ]:
bestfit = deepcopy(spec_resamp)
bestfit.flux = ppxf_best.bestfit

## Resampling the residuals

In [ ]:
def resample_residuals(bestfit, resid):
    newspec = deepcopy(bestfit)
    newspec.flux += resid * np.random.normal(size=len(resid))
    return newspec

In [ ]:
sedmodel_bootstrap =  desi_utils.fit_continuum.SEDModel(
    sps_file = "spectra_fsps_9.0.npz",
    regul_err = 0
)

In [ ]:
N_samples = 100

In [ ]:
resample_residuals(bestfit, resid).plot()
plt.yscale("log")

In [ ]:
sedmodel_bootstrap.load_templates(spec_resamp)

In [ ]:
bootstrap_fits = []
N_samples = 100

for i in range(N_samples):
    print(i)
    fit = sedmodel_bootstrap.fit(resample_residuals(bestfit, resid))
    bootstrap_fits.append(fit)